# Витрина транзакций ТЭ — 7. Клиент против своей группы

AI-14135. Задача тетрадки — найти клиентов, которые заметно отличаются от похожих на них, и подсветить их на проверку. Период — август–октябрь 2026, все операции клиента за три месяца вместе.

Похожие — это клиенты той же группы в одном из четырёх разрезов: код точки (MCC), MCC + регион (Красноярский край, Москва, прочие), основной ОКВЭД, MCC + размер бизнеса. Клиент сравнивается с остальными клиентами группы, сам он в норму группы не входит.

Тесты — те, что указаны в задаче:
- Колмогоров–Смирнов — устроены ли чеки клиента иначе, чем в группе;
- Манн–Уитни — систематически ли чеки клиента крупнее или мельче;
- хи-квадрат — в другие ли часы суток идут операции;
- косинус — похож ли портрет клиента в целом на типичный портрет группы.

Вход: профиль `sandbox_ai.tmp_shestopalov_te_profile` (одна строка на ИНН) и агрегат по факту `sandbox_ai.tmp_shestopalov_acq_trx`: сколько операций у клиента в каждой корзине суммы чека и в каждом часе суток. Сами операции в тетрадку не загружаются. Клиенты без кода точки (`mcc_code = не заполнен`) убираются: это пустой код, не вид деятельности.

Кроме проверки, тетрадка выгружает рейтинги регионов и кодов точки, численность групп и широкую таблицу клиентов.


In [ ]:
from IPython.display import display
from rail_connectors.connection import connect

imp = connect(
    to="IMPALA",
    extra_options={"db": "sandbox_ai"},
    driver_args={"tez.queue.name": "ai"},
    kerberos={
        "keytab_path": "/home/jovyan/test_requests/tech.keytab",
        "use_credentials": True,
        "update_keytab": True,
    },
    user_params={"user_name": "Shestopalov-VYur"},
)
imp._init_connection()


In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

MIN_CLIENTS = 30
COMFORT_CLIENTS = 100

SLICES = [
    ("MCC", "mcc_code"),
    ("MCC + регион", "slice_mcc_region"),
    ("ОКВЭД", "okved"),
    ("MCC + размер бизнеса", "slice_mcc_size"),
]


def enough_label(n: int) -> str:
    if n >= COMFORT_CLIENTS:
        return "достаточно"
    if n >= MIN_CLIENTS:
        return "на грани"
    return "мало"


def coverage_table(profile: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for slice_name, col in SLICES:
        if col not in profile.columns:
            continue
        counts = profile[col].fillna("не заполнен").astype(str).value_counts()
        total = int(counts.sum())
        for value, n in counts.items():
            rows.append(
                {
                    "slice_name": slice_name,
                    "slice_value": value,
                    "client_cnt": int(n),
                    "client_share": int(n) / total if total else np.nan,
                    "enough": enough_label(int(n)),
                }
            )
    if not rows:
        return pd.DataFrame(columns=["slice_name", "slice_value", "client_cnt", "client_share", "enough"])
    return pd.DataFrame(rows).sort_values(["slice_name", "client_cnt"], ascending=[True, False])


def bh_adjust(pvalues: pd.Series) -> pd.Series:
    p = pvalues.to_numpy(dtype=float)
    out = np.full(len(p), np.nan)
    mask = np.isfinite(p)
    if mask.sum() == 0:
        return pd.Series(out, index=pvalues.index)
    pv = p[mask]
    n = len(pv)
    order = np.argsort(pv)
    ranked = pv[order]
    adj = ranked * n / np.arange(1, n + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0, 1)
    placed = np.empty(n)
    placed[order] = adj
    out[np.flatnonzero(mask)] = placed
    return pd.Series(out, index=pvalues.index)


with imp:
    profile = imp.fetch("select * from sandbox_ai.tmp_shestopalov_te_profile")

profile["inn"] = profile["inn"].astype(str)
profile_all = profile.copy()
before = len(profile)
profile = profile.loc[profile["mcc_code"].astype(str) != "не заполнен"].copy()
print(f"Клиентов в профиле: {before}. Без кода точки убрано: {before - len(profile)}. В расчёте: {len(profile)}.")
profile.head()


## Названия кодов

Рядом с цифровым кодом выводится название: MCC точки, ОКВЭД и регион регистрации по ОКАТО. Если полного кода ОКВЭД нет в справочнике, пишется ближайший верхний код, в который он входит. Слова в группе MCC (аптека, гостиница, бар) уже названия, их не переводим. Ячейка печатает коды точки, которых нет в справочнике.


In [ ]:
MCC_NAMES = {
    "742": "ветеринария",
    "763": "сельхозкооперативы",
    "780": "ландшафтный дизайн",
    "1520": "генподрядчики, строительство",
    "1711": "сантехника и отопление, подряд",
    "1731": "электромонтаж",
    "1799": "прочие строительные подрядчики",
    "4111": "пассажирский транспорт",
    "4121": "такси",
    "4131": "автобусные перевозки",
    "4214": "грузоперевозки",
    "4215": "курьерские услуги",
    "4722": "турагентства",
    "4784": "платные дороги",
    "4812": "телефоны и связь, оборудование",
    "4814": "услуги связи",
    "4816": "интернет, компьютерные сети",
    "4899": "кабельное и платное ТВ",
    "4900": "коммунальные услуги",
    "5013": "автозапчасти, опт",
    "5039": "стройматериалы, опт",
    "5044": "офисное оборудование",
    "5045": "компьютеры, опт",
    "5047": "медицинское оборудование",
    "5065": "электротовары, опт",
    "5072": "скобяные товары, опт",
    "5074": "сантехника, опт",
    "5085": "промышленные товары, опт",
    "5099": "товары длительного пользования, опт",
    "5111": "канцтовары, опт",
    "5122": "лекарства, опт",
    "5131": "ткани, опт",
    "5137": "спецодежда, опт",
    "5169": "химия, опт",
    "5172": "нефтепродукты, опт",
    "5192": "книги и печать, опт",
    "5193": "цветы и растения, опт",
    "5199": "прочие товары, опт",
    "5200": "товары для дома, крупный формат",
    "5211": "стройматериалы",
    "5231": "краски, стекло, обои",
    "5251": "скобяные товары, инструменты",
    "5261": "сад и огород",
    "5300": "оптовые клубы",
    "5309": "беспошлинные магазины",
    "5310": "дискаунтеры",
    "5311": "универмаги",
    "5331": "универсальные магазины",
    "5399": "товары смешанного ассортимента",
    "5411": "супермаркет",
    "5422": "мясо и рыба",
    "5441": "кондитерская",
    "5451": "молочный магазин",
    "5462": "пекарня",
    "5499": "прочие продуктовые",
    "5511": "автосалон",
    "5521": "подержанные автомобили",
    "5531": "автомагазин",
    "5532": "шины",
    "5533": "автозапчасти",
    "5541": "АЗС",
    "5542": "автоматическая АЗС",
    "5551": "лодки",
    "5571": "мотоциклы",
    "5599": "прочая автомототехника",
    "5611": "мужская одежда",
    "5621": "женская одежда",
    "5631": "женские аксессуары",
    "5641": "детская одежда",
    "5651": "одежда для всей семьи",
    "5655": "спортивная одежда",
    "5661": "обувь",
    "5681": "меха",
    "5691": "одежда мужская и женская",
    "5697": "ателье, пошив",
    "5699": "прочая одежда и аксессуары",
    "5712": "мебель",
    "5713": "напольные покрытия",
    "5714": "шторы и ткани для дома",
    "5719": "товары для дома",
    "5722": "бытовая техника",
    "5732": "электроника",
    "5733": "музыкальные инструменты",
    "5734": "программное обеспечение",
    "5735": "музыкальные записи",
    "5811": "кейтеринг",
    "5812": "ресторан",
    "5813": "бар",
    "5814": "фастфуд",
    "5815": "цифровые книги и фильмы",
    "5816": "цифровые игры",
    "5817": "цифровые приложения",
    "5818": "цифровые товары",
    "5912": "аптека",
    "5921": "алкоголь",
    "5931": "секонд-хенд",
    "5932": "антиквариат",
    "5933": "ломбард",
    "5940": "велосипеды",
    "5941": "спорттовары",
    "5942": "книжный магазин",
    "5943": "канцтовары",
    "5944": "ювелирный магазин, часы",
    "5945": "игрушки",
    "5946": "фототовары",
    "5947": "подарки и сувениры",
    "5948": "кожгалантерея",
    "5949": "ткани и швейные принадлежности",
    "5950": "посуда",
    "5970": "товары для творчества",
    "5971": "художественные галереи",
    "5973": "религиозные товары",
    "5975": "слуховые аппараты",
    "5976": "ортопедия",
    "5977": "косметика",
    "5983": "топливо: уголь, газ в баллонах",
    "5992": "цветы",
    "5993": "табак",
    "5994": "газеты и журналы",
    "5995": "зоотовары",
    "5996": "бассейны",
    "5998": "палатки и тенты",
    "5999": "прочие специализированные магазины",
    "6010": "выдача наличных в отделении",
    "6011": "банкомат",
    "6012": "финансовые услуги",
    "6051": "валюта, переводы",
    "6300": "страхование",
    "6513": "аренда недвижимости",
    "7011": "гостиница",
    "7032": "спортивные и детские лагеря",
    "7033": "кемпинги",
    "7210": "прачечные",
    "7216": "химчистка",
    "7221": "фотостудия",
    "7230": "парикмахерская, салон красоты",
    "7251": "ремонт обуви",
    "7261": "ритуальные услуги",
    "7277": "консультации, психологи",
    "7296": "прокат одежды",
    "7297": "массаж",
    "7298": "спа, оздоровление",
    "7299": "прочие бытовые услуги",
    "7311": "реклама",
    "7333": "фото и графический дизайн",
    "7338": "копирование и печать",
    "7342": "дезинсекция",
    "7349": "уборка",
    "7361": "кадровые агентства",
    "7372": "программирование",
    "7379": "ремонт компьютеров",
    "7392": "консалтинг",
    "7393": "охрана",
    "7394": "прокат оборудования",
    "7395": "фотолаборатория",
    "7399": "прочие бизнес-услуги",
    "7512": "прокат автомобилей",
    "7523": "парковка",
    "7531": "кузовной ремонт",
    "7534": "шиномонтаж",
    "7535": "покраска автомобилей",
    "7538": "автосервис",
    "7542": "автомойка",
    "7549": "эвакуатор",
    "7622": "ремонт электроники",
    "7623": "ремонт кондиционеров",
    "7629": "ремонт бытовой техники",
    "7631": "ремонт часов и ювелирных изделий",
    "7641": "перетяжка мебели",
    "7692": "сварка",
    "7699": "прочий ремонт",
    "7832": "кинотеатр",
    "7911": "танцевальная студия",
    "7922": "театр, билеты",
    "7929": "музыканты",
    "7932": "бильярд",
    "7933": "боулинг",
    "7941": "спортивные клубы, стадионы",
    "7991": "достопримечательности",
    "7993": "игровые автоматы",
    "7994": "компьютерный клуб",
    "7996": "парк развлечений",
    "7997": "фитнес, клубы",
    "7998": "зоопарк, аквариум",
    "7999": "прочие развлечения",
    "8011": "врачи",
    "8021": "стоматология",
    "8031": "остеопаты",
    "8041": "мануальная терапия",
    "8042": "окулисты",
    "8043": "оптика",
    "8049": "подологи",
    "8050": "сиделки, уход",
    "8062": "больница",
    "8071": "медицинская лаборатория",
    "8099": "прочие медицинские услуги",
    "8111": "юристы",
    "8211": "школы",
    "8220": "вузы",
    "8241": "заочное обучение",
    "8244": "бизнес-школы",
    "8249": "профобучение",
    "8299": "курсы, прочее образование",
    "8351": "детский сад",
    "8398": "благотворительность",
    "8641": "общественные организации",
    "8661": "религиозные организации",
    "8699": "членские организации",
    "8734": "лаборатории",
    "8911": "архитекторы",
    "8931": "бухгалтерия",
    "8999": "прочие профессиональные услуги",
    "9222": "штрафы",
    "9311": "налоги",
    "9399": "госуслуги",
    "9402": "почта",
}

OKVED_NAMES = {
    "01": "растениеводство и животноводство",
    "02": "лесоводство и лесозаготовки",
    "03": "рыболовство и рыбоводство",
    "05": "добыча угля",
    "06": "добыча нефти и газа",
    "07": "добыча металлических руд",
    "08": "добыча прочих полезных ископаемых",
    "09": "услуги в добыче",
    "10": "производство пищевых продуктов",
    "11": "производство напитков",
    "12": "производство табачных изделий",
    "13": "производство текстиля",
    "14": "производство одежды",
    "15": "производство кожи и изделий из кожи",
    "16": "обработка древесины",
    "17": "производство бумаги",
    "18": "полиграфия",
    "19": "кокс и нефтепродукты",
    "20": "химическое производство",
    "21": "производство лекарств",
    "22": "резиновые и пластмассовые изделия",
    "23": "неметаллическая минеральная продукция",
    "24": "металлургия",
    "25": "готовые металлические изделия",
    "26": "компьютеры, электроника, оптика",
    "27": "электрическое оборудование",
    "28": "машины и оборудование",
    "29": "автотранспортные средства",
    "30": "прочие транспортные средства",
    "31": "производство мебели",
    "32": "прочие готовые изделия",
    "33": "ремонт и монтаж машин",
    "35": "электроэнергия, газ, пар",
    "36": "водоснабжение",
    "37": "сточные воды",
    "38": "сбор и утилизация отходов",
    "39": "рекультивация",
    "41": "строительство зданий",
    "42": "строительство инженерных сооружений",
    "43": "специализированные строительные работы",
    "45": "торговля и ремонт автомобилей",
    "46": "оптовая торговля",
    "47": "розничная торговля",
    "49": "сухопутный транспорт",
    "50": "водный транспорт",
    "51": "воздушный транспорт",
    "52": "склады и вспомогательная транспортная деятельность",
    "53": "почта и курьеры",
    "55": "временное проживание",
    "56": "общепит",
    "58": "издательская деятельность",
    "59": "кино, видео, звукозапись",
    "60": "теле- и радиовещание",
    "61": "связь",
    "62": "разработка ПО",
    "63": "информационные технологии",
    "64": "финансовые услуги",
    "65": "страхование",
    "66": "вспомогательные финансовые услуги",
    "68": "операции с недвижимостью",
    "69": "право и бухучёт",
    "70": "управление, консалтинг",
    "71": "архитектура и инженерия",
    "72": "научные исследования",
    "73": "реклама и исследование рынка",
    "74": "прочая профессиональная деятельность",
    "75": "ветеринария",
    "77": "аренда и лизинг",
    "78": "подбор персонала",
    "79": "туристические агентства",
    "80": "охрана и расследования",
    "81": "обслуживание зданий",
    "82": "административно-хозяйственные услуги",
    "84": "госуправление",
    "85": "образование",
    "86": "здравоохранение",
    "87": "уход с проживанием",
    "88": "социальные услуги",
    "90": "творчество и искусство",
    "91": "библиотеки, музеи",
    "92": "азартные игры",
    "93": "спорт, отдых, развлечения",
    "94": "общественные организации",
    "95": "ремонт компьютеров и бытовых вещей",
    "96": "прочие личные услуги",
    "01.11": "зерновые и масличные",
    "01.11.1": "выращивание зерновых",
    "01.13": "овощи и корнеплоды",
    "01.41": "молочный скот",
    "01.45": "овцы и козы",
    "01.46": "свиньи",
    "01.47": "птица",
    "01.50": "смешанное сельское хозяйство",
    "10.11": "переработка мяса",
    "10.13": "мясные продукты",
    "10.51": "молочная продукция",
    "10.71": "хлеб и недолговечная выпечка",
    "10.89": "прочие пищевые продукты",
    "45.20": "ремонт автомобилей",
    "45.32": "розница автозапчастями",
    "46.90": "неспециализированный опт",
    "47.1": "неспециализированная розница",
    "47.11": "продовольственный магазин",
    "47.19": "прочий неспециализированный магазин",
    "47.2": "специализированная продуктовая розница",
    "47.21": "овощи и фрукты",
    "47.22": "мясо",
    "47.23": "рыба",
    "47.24": "хлеб и кондитерские изделия",
    "47.25": "напитки",
    "47.26": "табак",
    "47.29": "прочие продукты",
    "47.30": "АЗС, моторное топливо",
    "47.4": "компьютеры и связь",
    "47.5": "бытовые товары",
    "47.51": "текстиль",
    "47.52": "скобяные товары, краски, стекло",
    "47.53": "ковры и обои",
    "47.54": "бытовая электротехника",
    "47.59": "мебель и прочие бытовые товары",
    "47.6": "товары для досуга",
    "47.61": "книги",
    "47.62": "газеты и канцтовары",
    "47.64": "спорттовары",
    "47.65": "игрушки",
    "47.7": "прочие специализированные магазины",
    "47.71": "одежда",
    "47.72": "обувь и кожаные изделия",
    "47.73": "аптека",
    "47.74": "медицинские и ортопедические товары",
    "47.75": "косметика",
    "47.76": "цветы, семена, животные и корма",
    "47.76.2": "домашние животные и корма",
    "47.77": "часы и ювелирные изделия",
    "47.78": "прочие специализированные товары",
    "47.79": "подержанные товары",
    "47.8": "рынки и киоски",
    "47.81": "продукты на рынках и в киосках",
    "47.82": "одежда и обувь на рынках",
    "47.89": "прочие товары на рынках",
    "47.9": "розница вне магазинов",
    "47.91": "розница через интернет и по почте",
    "47.99": "прочая розница вне магазинов",
    "49.32": "такси",
    "49.41": "грузовые автоперевозки",
    "55.10": "гостиница",
    "55.20": "краткосрочное проживание",
    "56.10": "рестораны и доставка еды",
    "56.21": "кейтеринг",
    "56.29": "столовые",
    "56.30": "бары",
    "68.20": "аренда своей недвижимости",
    "68.31": "агентства недвижимости",
    "85.41": "дополнительное образование",
    "86.10": "больницы",
    "86.21": "общая врачебная практика",
    "86.22": "специальная врачебная практика",
    "86.23": "стоматология",
    "86.90": "прочая медицина",
    "93.11": "спортивные объекты",
    "93.13": "фитнес-центры",
    "93.29": "прочие развлечения",
    "96.01": "стирка и химчистка",
    "96.02": "парикмахерские и салоны красоты",
    "96.02.1": "парикмахерские",
    "96.02.2": "косметические услуги",
    "96.03": "ритуальные услуги",
    "96.04": "бани, сауны",
    "96.09": "прочие личные услуги",
}

REGION_NAMES = {
    "01": "Алтайский край",
    "03": "Краснодарский край",
    "04": "Красноярский край",
    "05": "Приморский край",
    "07": "Ставропольский край",
    "08": "Хабаровский край",
    "10": "Амурская обл.",
    "11": "Архангельская обл.",
    "12": "Астраханская обл.",
    "14": "Белгородская обл.",
    "15": "Брянская обл.",
    "17": "Владимирская обл.",
    "18": "Волгоградская обл.",
    "19": "Вологодская обл.",
    "20": "Воронежская обл.",
    "22": "Нижегородская обл.",
    "24": "Ивановская обл.",
    "25": "Иркутская обл.",
    "26": "Ингушетия",
    "27": "Калининградская обл.",
    "28": "Тверская обл.",
    "29": "Калужская обл.",
    "30": "Камчатский край",
    "32": "Кемеровская обл.",
    "33": "Кировская обл.",
    "34": "Костромская обл.",
    "35": "Крым",
    "36": "Самарская обл.",
    "37": "Курганская обл.",
    "38": "Курская обл.",
    "40": "Санкт-Петербург",
    "41": "Ленинградская обл.",
    "42": "Липецкая обл.",
    "44": "Магаданская обл.",
    "45": "Москва",
    "46": "Московская обл.",
    "47": "Мурманская обл.",
    "49": "Новгородская обл.",
    "50": "Новосибирская обл.",
    "52": "Омская обл.",
    "53": "Оренбургская обл.",
    "54": "Орловская обл.",
    "56": "Пензенская обл.",
    "57": "Пермский край",
    "58": "Псковская обл.",
    "60": "Ростовская обл.",
    "61": "Рязанская обл.",
    "63": "Саратовская обл.",
    "64": "Сахалинская обл.",
    "65": "Свердловская обл.",
    "66": "Смоленская обл.",
    "67": "Севастополь",
    "68": "Тамбовская обл.",
    "69": "Томская обл.",
    "70": "Тульская обл.",
    "71": "Тюменская обл.",
    "73": "Ульяновская обл.",
    "75": "Челябинская обл.",
    "76": "Забайкальский край",
    "77": "Чукотский АО",
    "78": "Ярославская обл.",
    "79": "Адыгея",
    "80": "Башкортостан",
    "81": "Бурятия",
    "82": "Дагестан",
    "83": "Кабардино-Балкария",
    "84": "Республика Алтай",
    "85": "Калмыкия",
    "86": "Карелия",
    "87": "Коми",
    "88": "Марий Эл",
    "89": "Мордовия",
    "90": "Северная Осетия",
    "91": "Карачаево-Черкесия",
    "92": "Татарстан",
    "93": "Тыва",
    "94": "Удмуртия",
    "95": "Хакасия",
    "96": "Чечня",
    "97": "Чувашия",
    "98": "Якутия",
    "99": "Еврейская АО",
}


def mcc_label(value):
    text = str(value).strip()
    if not text.isdigit():
        return text
    name = MCC_NAMES.get(text.lstrip("0") or "0")
    return f"{text} {name}" if name else f"{text} (нет в справочнике)"


def okved_label(value):
    text = str(value).strip()
    if text in ("", "не заполнен"):
        return text
    key = text
    while key:
        if key in OKVED_NAMES:
            if key == text:
                return f"{text} {OKVED_NAMES[key]}"
            return f"{text} (входит в {key} {OKVED_NAMES[key]})"
        key = key.rsplit(".", 1)[0] if "." in key else ""
    return f"{text} (нет в справочнике)"


def region_label(value):
    text = str(value).strip()
    name = REGION_NAMES.get(text)
    return f"{text} {name}" if name else text


SLICE_COLS = dict(SLICES)


def value_label(slice_name, value):
    col = SLICE_COLS.get(slice_name)
    text = str(value)
    if col in ("mcc_code", "mcc_group"):
        return mcc_label(text)
    if col == "slice_mcc_region":
        code, _, region = text.partition(" | ")
        return f"{mcc_label(code)} | {region_label(region)}"
    if col == "slice_mcc_size":
        code, _, size = text.partition(" | ")
        return f"{mcc_label(code)} | {size}"
    if col in ("okved", "okved_class"):
        return okved_label(text)
    return text


missing = sorted(
    {
        str(v)
        for v in profile["mcc_code"].astype(str).unique()
        if v.isdigit() and (v.lstrip("0") or "0") not in MCC_NAMES
    }
)
print("Кодов точки без названия в справочнике:", len(missing), missing[:30])


## Численность групп

`достаточно` — от 100 клиентов, `на грани` — от 30 до 99, `мало` — меньше 30. Клиентов проверяем только в группах от 30 клиентов: в меньшей группе норма ненадёжна. В тестах дополнительно не участвуют клиенты меньше чем с 30 операциями, поэтому проверенных групп может оказаться чуть меньше, чем здесь.


In [ ]:
coverage = coverage_table(profile)
coverage.insert(2, "name", [value_label(s, v) for s, v in zip(coverage["slice_name"], coverage["slice_value"])])
summary = (
    coverage.groupby(["slice_name", "enough"], as_index=False)
    .agg(groups=("slice_value", "count"), clients=("client_cnt", "sum"))
    .sort_values(["slice_name", "enough"])
)
display(summary)
usable = coverage.loc[coverage["enough"] != "мало"].copy()
print(f"Групп, где клиентов не меньше 30: {len(usable)}.")
with pd.option_context("display.max_rows", 200, "display.max_colwidth", 80):
    display(usable[["slice_name", "name", "client_cnt", "client_share", "enough"]])


## Операции клиентов по суммам и часам

Один запрос к факту за три месяца, без отменённых операций (статус `R`). Результат — сколько операций у каждого клиента в каждой корзине суммы чека и в каждом часе суток. Корзины сумм в рублях: до 100, 100–200, 200–300, 300–500, 500–700, 700–1000, 1–1,5 тыс., 1,5–2, 2–3, 3–5, 5–7, 7–10, 10–15, 15–20, 20–30, 30–50, 50–100 тыс., от 100 тыс. Тот же SQL лежит в `sql/07_te_client_hist.sql`.


In [ ]:
AMOUNT_LABELS = [
    "до 100", "100-200", "200-300", "300-500", "500-700", "700-1000",
    "1000-1500", "1500-2000", "2000-3000", "3000-5000", "5000-7000", "7000-10000",
    "10-15 тыс", "15-20 тыс", "20-30 тыс", "30-50 тыс", "50-100 тыс", "от 100 тыс",
]

HIST_SQL = """
select inn, kind, bin, count(*) as cnt
from (
  select
    inn,
    'amount' as kind,
    cast(case
      when sum_t < 100 then 0
      when sum_t < 200 then 1
      when sum_t < 300 then 2
      when sum_t < 500 then 3
      when sum_t < 700 then 4
      when sum_t < 1000 then 5
      when sum_t < 1500 then 6
      when sum_t < 2000 then 7
      when sum_t < 3000 then 8
      when sum_t < 5000 then 9
      when sum_t < 7000 then 10
      when sum_t < 10000 then 11
      when sum_t < 15000 then 12
      when sum_t < 20000 then 13
      when sum_t < 30000 then 14
      when sum_t < 50000 then 15
      when sum_t < 100000 then 16
      else 17
    end as int) as bin
  from sandbox_ai.tmp_shestopalov_acq_trx
  where status <> 'R'
) a
group by inn, kind, bin
union all
select inn, kind, bin, count(*) as cnt
from (
  select inn, 'hour' as kind, cast(hour(date_t) as int) as bin
  from sandbox_ai.tmp_shestopalov_acq_trx
  where status <> 'R'
) h
group by inn, kind, bin
"""


def hist_matrix(hist, kind, n_bins):
    part = hist.loc[hist["kind"] == kind]
    matrix = part.pivot_table(index="inn", columns="bin", values="cnt", aggfunc="sum", fill_value=0)
    return matrix.reindex(columns=range(n_bins), fill_value=0).astype(float)


with imp:
    hist = imp.fetch(HIST_SQL)

hist["inn"] = hist["inn"].astype(str)
hist["bin"] = pd.to_numeric(hist["bin"]).astype(int)
hist["cnt"] = pd.to_numeric(hist["cnt"])
amount = hist_matrix(hist, "amount", len(AMOUNT_LABELS))
hours = hist_matrix(hist, "hour", 24)
print(f"Клиентов с операциями: {len(amount)}. Операций: {int(amount.to_numpy().sum())}.")


## Тесты: клиент против своей группы

Для каждой группы от 30 клиентов норма — среднее распределение остальных клиентов группы, у каждого клиента одинаковый вес. Так один гигант с миллионом чеков не задаёт норму, а сам проверяемый клиент не влияет на то, с чем его сравнивают. Клиент меньше чем с 30 операциями не проверяется.

**Колмогоров–Смирнов, суммы чеков.** Для каждой суммы считаем, какая доля чеков ниже неё у клиента и в норме. `ks_d` — самый большой разрыв между этими долями, от 0 до 1. На пальцах: у фастфудов 90% чеков до 1000 рублей, а у клиента до 1000 рублей только 20%. Разрыв 0,7 — чеки устроены совсем иначе. Тест ловит то, что не видно по среднему: пачку крупных чеков, чеки одной суммы.

**Манн–Уитни, суммы чеков.** Берём случайный чек клиента и случайный чек группы. `mw_доля_пар` — как часто чек клиента больше. 0,5 — как у всех, 0,9 — у клиента почти всегда крупнее, 0,1 — почти всегда мельче. Колмогоров–Смирнов говорит «распределение другое», Манн–Уитни — «в какую сторону».

**Хи-квадрат, часы суток.** Сравниваем, сколько операций клиента пришлось на каждый час и сколько было бы, если бы он работал по часам как группа. Сила отличия — V Крамера `chi_v`: около 0 — как группа, от 0,3 — заметно иначе. Рядом пишется час с самым большим перекосом: «в 02–03 ч 18% операций против 1% в группе». Часы, где ожидается меньше 5 операций, склеиваются с соседними, иначе тест врёт.

**Косинус, портрет.** Портрет — 9 метрик профиля: операций, оборот, средний и медианный чек, медианный интервал, активные дни, доля успешных, доля расхождений MCC, операций на BIN. Метрики приводятся к одному масштабу: суммы и количества через логарифм, затем z по всем клиентам. Косинус сравнивает направление портрета клиента с типичным портретом группы: близко к 1 — похож, около 0 и ниже — устроен иначе. В разрезе «MCC + размер» при `business_size_is_proxy = 1` операции и оборот в портрет не входят: размер собран из них.

**Значимость и флаги.** На тысячах операций p-значение почти у всех клиентов крошечное: любое мелкое отличие «значимо». Поэтому p поправляется по Бенджамини–Хохбергу внутри разреза и теста и служит только фильтром, а флаг ставится по силе отличия:

| тест | флаг, если |
|---|---|
| Колмогоров–Смирнов | `ks_d >= 0,3` и поправленное p < 0,01 |
| Манн–Уитни | `mw_доля_пар >= 0,8` или `<= 0,2`, и поправленное p < 0,01 |
| хи-квадрат | `chi_v >= 0,3` и поправленное p < 0,01 |
| косинус | `косинус <= 0` |

В каждом разрезе у клиента `флагов` от 0 до 4. По клиенту в целом — `разрезов_с_флагами`, наибольшее `флагов` и `что_выбивается` словами. Пороги — константы вверху ячейки расчёта: если подсвечено слишком много клиентов, их можно поднять.

Оговорки:
- операции одного клиента не независимы (одни и те же покупатели, повторяющиеся суммы), поэтому p оптимистичны — опираемся на силу отличия;
- если время операций записано в одном часовом поясе (не проверялось), часы честнее сравнивать в разрезе «MCC + регион»: у Красноярского края разница с Москвой 4 часа;
- флаг — повод посмотреть клиента, а не вывод о фроде. Крупная сеть или клиент с необычным, но законным форматом тоже может выбиться.


In [ ]:
TEST_MIN_OPS = 30
P_MAX = 0.01
KS_MIN = 0.3
MW_HIGH = 0.8
MW_LOW = 0.2
CHI_V_MIN = 0.3
COS_MAX = 0.0

PORTRAIT_FIELDS = [
    "tx_count", "turnover", "avg_amount", "median_amount", "median_interval",
    "active_days", "success_rate", "mcc_mismatch_share", "ops_per_bin",
]
SHARE_FIELDS = {"success_rate", "mcc_mismatch_share"}
# Размер бизнеса-прокси собран из этих двух метрик.
SIZE_BUILT_FROM = {"tx_count", "turnover"}


def _num(value, digits=2):
    return f"{value:.{digits}f}".replace(".", ",")


def portrait_matrix(profile):
    work = profile.copy()
    for col in set(PORTRAIT_FIELDS) - {"ops_per_bin"} | {"unique_bins"}:
        work[col] = pd.to_numeric(work[col], errors="coerce")
    work["ops_per_bin"] = work["tx_count"] / work["unique_bins"].where(work["unique_bins"] > 0)
    out = pd.DataFrame(index=work["inn"].to_numpy())
    for col in PORTRAIT_FIELDS:
        x = work[col].to_numpy(dtype=float)
        if col not in SHARE_FIELDS:
            x = np.log1p(np.clip(x, 0, None))
        out[col] = x
    out = out.fillna(out.median())
    out = out.loc[:, out.std(ddof=0) > 0]
    return (out - out.mean()) / out.std(ddof=0)


def _chi_hours(obs, exp):
    merged_o, merged_e = [], []
    acc_o = acc_e = 0.0
    for o, e in zip(obs, exp):
        acc_o += o
        acc_e += e
        if acc_e >= 5:
            merged_o.append(acc_o)
            merged_e.append(acc_e)
            acc_o = acc_e = 0.0
    if acc_o or acc_e:
        if merged_e:
            merged_o[-1] += acc_o
            merged_e[-1] += acc_e
        else:
            merged_o.append(acc_o)
            merged_e.append(acc_e)
    k = len(merged_e)
    n = float(np.sum(obs))
    if k < 2 or n == 0:
        return np.nan, np.nan
    o = np.array(merged_o)
    e = np.array(merged_e)
    chi2 = float(((o - e) ** 2 / e).sum())
    return np.sqrt(chi2 / (n * (k - 1))), float(stats.chi2.sf(chi2, k - 1))


def _group_tests(members, amount, hours, portrait, cos_cols):
    """Каждый клиент группы против нормы остальных; у каждого клиента нормы одинаковый вес."""
    k = len(members)
    a = amount.loc[members].to_numpy()
    n = a.sum(axis=1)
    p = a / n[:, None]
    norm = (p.sum(axis=0) - p) / (k - 1)
    cdf_c = p.cumsum(axis=1)
    cdf_n = norm.cumsum(axis=1)

    ks_d = np.abs(cdf_c - cdf_n).max(axis=1)
    ks_p = stats.kstwo.sf(ks_d, n.astype(int))

    auc = (p * (cdf_n - 0.5 * norm)).sum(axis=1)
    n_rest = n.sum() - n
    sd = np.sqrt(n * n_rest * (n + n_rest + 1) / 12)
    mw_p = 2 * stats.norm.sf(np.abs(auc - 0.5) * n * n_rest / sd)

    h = hours.reindex(members).fillna(0).to_numpy()
    nh = h.sum(axis=1)
    q = h / np.where(nh > 0, nh, 1)[:, None]
    norm_h = (q.sum(axis=0) - q) / (k - 1)
    chi = [_chi_hours(h[i], nh[i] * norm_h[i]) for i in range(k)]
    top = (q - norm_h).argmax(axis=1)
    top_text = [
        f"в {t:02d}–{t + 1:02d} ч {q[i, t] * 100:.0f}% операций против {norm_h[i, t] * 100:.0f}% в группе"
        for i, t in enumerate(top)
    ]

    z = portrait.reindex(members)[cos_cols].to_numpy()
    rest = (z.sum(axis=0) - z) / (k - 1)
    den = np.linalg.norm(z, axis=1) * np.linalg.norm(rest, axis=1)
    cos = np.where(den > 0, (z * rest).sum(axis=1) / np.where(den > 0, den, 1), np.nan)

    return pd.DataFrame(
        {
            "inn": members,
            "операций": n.astype(int),
            "ks_d": ks_d,
            "ks_p": ks_p,
            "mw_доля_пар": auc,
            "mw_p": mw_p,
            "chi_v": [c[0] for c in chi],
            "chi_p": [c[1] for c in chi],
            "главный_час": top_text,
            "косинус": cos,
        }
    )


def client_tests(profile, amount, hours, portrait):
    ops = amount.sum(axis=1)
    proxy = profile.get("business_size_is_proxy")
    size_is_proxy = proxy is not None and pd.to_numeric(proxy, errors="coerce").fillna(0).eq(1).any()
    pieces = []
    for slice_name, col in SLICES:
        labels = profile[col].fillna("не заполнен").astype(str)
        cos_cols = list(portrait.columns)
        if size_is_proxy and col == "slice_mcc_size":
            cos_cols = [c for c in cos_cols if c not in SIZE_BUILT_FROM]
        for value, inns in profile["inn"].groupby(labels):
            if value in ("не заполнен", "нет оборота"):
                continue
            members = [m for m in inns if ops.get(m, 0) >= TEST_MIN_OPS]
            if len(members) < MIN_CLIENTS:
                continue
            part = _group_tests(members, amount, hours, portrait, cos_cols)
            part.insert(0, "разрез", slice_name)
            part.insert(1, "группа", value)
            part.insert(2, "группа_название", value_label(slice_name, value))
            part.insert(3, "клиентов_в_группе", len(members))
            pieces.append(part)
    tests = pd.concat(pieces, ignore_index=True)
    for name in ("ks", "mw", "chi"):
        tests[f"{name}_p_bh"] = tests.groupby("разрез")[f"{name}_p"].transform(bh_adjust)

    tests["флаг_ks"] = ((tests["ks_p_bh"] < P_MAX) & (tests["ks_d"] >= KS_MIN)).astype(int)
    tests["флаг_mw"] = (
        (tests["mw_p_bh"] < P_MAX) & ((tests["mw_доля_пар"] >= MW_HIGH) | (tests["mw_доля_пар"] <= MW_LOW))
    ).astype(int)
    tests["флаг_chi"] = ((tests["chi_p_bh"] < P_MAX) & (tests["chi_v"] >= CHI_V_MIN)).astype(int)
    tests["флаг_cos"] = (tests["косинус"] <= COS_MAX).astype(int)
    tests["флагов"] = tests[["флаг_ks", "флаг_mw", "флаг_chi", "флаг_cos"]].sum(axis=1)

    notes = []
    for ks, mw, chi, cs, d, auc, hour_text, v, c in zip(
        tests["флаг_ks"], tests["флаг_mw"], tests["флаг_chi"], tests["флаг_cos"], tests["ks_d"],
        tests["mw_доля_пар"], tests["главный_час"], tests["chi_v"], tests["косинус"],
    ):
        parts = []
        if mw:
            if auc >= 0.5:
                parts.append(f"чеки крупнее, чем в группе, в {auc * 100:.0f}% пар")
            else:
                parts.append(f"чеки мельче, чем в группе, в {(1 - auc) * 100:.0f}% пар")
        if ks:
            parts.append(f"распределение чеков расходится с группой на {_num(d)}")
        if chi:
            parts.append(f"{hour_text} (V = {_num(v)})")
        if cs:
            parts.append(f"портрет не похож на группу (косинус {_num(c)})")
        notes.append("; ".join(parts))
    tests["что_выбивается"] = notes
    return tests.sort_values(["флагов", "ks_d"], ascending=False).reset_index(drop=True)


def client_summary(tests):
    hit = tests.loc[tests["флагов"] > 0].copy()
    hit["_text"] = hit["разрез"] + " " + hit["группа_название"] + ": " + hit["что_выбивается"]
    out = hit.groupby("inn").agg(
        разрезов_с_флагами=("разрез", "nunique"),
        флагов=("флагов", "max"),
        max_ks_d=("ks_d", "max"),
        что_выбивается=("_text", " | ".join),
    )
    return out.sort_values(["разрезов_с_флагами", "флагов", "max_ks_d"], ascending=False).reset_index()


def group_norms(tests, amount, hours):
    rows = []
    for (slice_name, value, label), part in tests.groupby(["разрез", "группа", "группа_название"]):
        inns = part["inn"].tolist()
        a = amount.loc[inns].to_numpy()
        h = hours.reindex(inns).fillna(0).to_numpy()
        pa = (a / a.sum(axis=1, keepdims=True)).mean(axis=0)
        ph = (h / np.maximum(h.sum(axis=1, keepdims=True), 1)).mean(axis=0)
        row = {"разрез": slice_name, "группа": value, "группа_название": label, "клиентов": len(inns)}
        row.update({f"чек_{lab}": share for lab, share in zip(AMOUNT_LABELS, pa)})
        row.update({f"час_{i:02d}": share for i, share in enumerate(ph)})
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["разрез", "клиентов"], ascending=[True, False]).reset_index(drop=True)


portrait = portrait_matrix(profile)
tests = client_tests(profile, amount, hours, portrait)
flagged = client_summary(tests)
norms = group_norms(tests, amount, hours)

print(f"Проверок «клиент × разрез»: {len(tests)}. Клиентов проверено: {tests['inn'].nunique()} из {len(profile)}.")
by_slice = tests.groupby("разрез").agg(
    групп=("группа", "nunique"),
    проверено=("inn", "count"),
    флаг_ks=("флаг_ks", "sum"),
    флаг_mw=("флаг_mw", "sum"),
    флаг_chi=("флаг_chi", "sum"),
    флаг_cos=("флаг_cos", "sum"),
    от_2_флагов=("флагов", lambda s: int((s >= 2).sum())),
)
display(by_slice)
print(f"Клиентов хотя бы с одним флагом: {len(flagged)}. С флагами от 2 хотя бы в одном разрезе: {int((flagged['флагов'] >= 2).sum())}.")
with pd.option_context("display.max_colwidth", 250):
    display(flagged.head(30))


## Выгрузка в Excel

Два файла рядом с тетрадкой.

`te_distributions_2026_08_10.xlsx` — справочные листы:
- `регионы` — все регионы регистрации по ОКАТО: клиенты, операции, оборот, доли, место; средний и медианный оборот на клиента и место по среднему; доля оборота, которую даёт крупнейший клиент региона; отметка, идёт ли регион в разрезах отдельно (Красноярский край и Москва) или в «прочих»;
- `mcc` — все коды точки: клиенты, операции, оборот, доли, место; в какую группу MCC попал код и остался ли он отдельной группой;
- `численность` — все группы четырёх разрезов, сколько клиентов и годится ли группа для проверки.

`te_client_tests_2026_08_10.xlsx` — результат тестов:
- `подозрительные` — клиенты, у которых хотя бы в одном разрезе 2 флага и больше, с регионом, ОКВЭД, оборотом и пояснением словами;
- `все_проверки` — каждая пара «клиент × разрез»: сила отличия и поправленное p по каждому тесту, косинус, флаги;
- `норма_групп` — с чем сравнивали: по каждой группе доля операций в каждой корзине суммы и в каждом часе.

Если на сервере нет библиотеки для записи xlsx, те же таблицы пишутся в csv с такими же именами.


In [ ]:
from pathlib import Path


def rank_table(frame, col, label_fn):
    work = frame.copy()
    work["код"] = work[col].fillna("не заполнен").astype(str)
    work["tx_count"] = pd.to_numeric(work["tx_count"], errors="coerce")
    work["turnover"] = pd.to_numeric(work["turnover"], errors="coerce")
    out = work.groupby("код", as_index=False).agg(
        клиентов=("inn", "nunique"),
        операций=("tx_count", "sum"),
        оборот=("turnover", "sum"),
        медианный_оборот_на_клиента=("turnover", "median"),
    )
    out.insert(1, "название", out["код"].map(label_fn))
    out["доля_клиентов"] = out["клиентов"] / out["клиентов"].sum()
    out["доля_оборота"] = out["оборот"] / out["оборот"].sum()
    out["место_по_обороту"] = out["оборот"].rank(ascending=False, method="min").astype(int)
    out["средний_оборот_на_клиента"] = out["оборот"] / out["клиентов"]
    out["место_по_среднему_обороту"] = out["средний_оборот_на_клиента"].rank(ascending=False, method="min").astype(int)
    out["доля_оборота_крупнейшего_клиента"] = work.groupby("код")["turnover"].max().reindex(out["код"]).to_numpy() / out["оборот"]
    out = out.sort_values(["клиентов", "оборот"], ascending=False).reset_index(drop=True)
    out.insert(0, "место_по_клиентам", range(1, len(out) + 1))
    return out


def save_book(path, sheets):
    try:
        with pd.ExcelWriter(path) as writer:
            for sheet, frame in sheets.items():
                frame.to_excel(writer, sheet_name=sheet, index=False)
                ws = writer.sheets[sheet]
                if hasattr(ws, "auto_filter"):
                    ws.auto_filter.ref = ws.dimensions
                    ws.freeze_panes = "B2"
        print("Сохранено:", path)
    except ImportError:
        for sheet, frame in sheets.items():
            out_csv = path.with_name(f"{path.stem}_{sheet}.csv")
            frame.to_csv(out_csv, index=False, encoding="utf-8-sig", sep=";")
            print("Сохранено:", out_csv)


regions_out = rank_table(profile_all, "region", lambda c: REGION_NAMES.get(c, ""))
separate = set(profile_all.loc[profile_all["region_group"].astype(str) != "прочие", "region"].astype(str))
regions_out["отдельно_в_разрезе"] = regions_out["код"].isin(separate).map({True: "да", False: "нет, в «прочих»"})

mcc_out = rank_table(profile_all, "mcc_code", lambda c: MCC_NAMES.get(c.lstrip("0") or "0", ""))
groups = profile_all.assign(_c=profile_all["mcc_code"].astype(str)).groupby("_c")["mcc_group"].first()
mcc_out["группа_mcc"] = mcc_out["код"].map(groups)
mcc_out["отдельная_группа"] = (mcc_out["группа_mcc"].astype(str) == mcc_out["код"]).map({True: "да", False: "нет, склеен"})

print("Крупнейшие регионы по числу клиентов:")
display(regions_out.head(10))
print("Крупнейшие коды точки по числу клиентов:")
display(mcc_out.head(15))

save_book(
    Path.cwd() / "te_distributions_2026_08_10.xlsx",
    {"регионы": regions_out, "mcc": mcc_out, "численность": coverage},
)

card = profile[["inn", "region", "business_size", "mcc_code", "okved", "turnover", "tx_count"]].copy()
card["регион_название"] = card["region"].astype(str).map(lambda c: REGION_NAMES.get(c, ""))
card["mcc_название"] = card["mcc_code"].astype(str).map(lambda c: MCC_NAMES.get(c.lstrip("0") or "0", ""))
card["оквэд_название"] = card["okved"].map(okved_label)
lead = ["inn", "region", "регион_название", "mcc_code", "mcc_название", "okved", "оквэд_название",
        "business_size", "turnover", "tx_count"]
suspicious = flagged.loc[flagged["флагов"] >= 2].merge(card[lead], on="inn", how="left")
suspicious = suspicious[lead + [c for c in suspicious.columns if c not in lead]]

save_book(
    Path.cwd() / "te_client_tests_2026_08_10.xlsx",
    {"подозрительные": suspicious, "все_проверки": tests, "норма_групп": norms},
)


## Широкая таблица клиентов

Аналог витрины в Excel: одна строка на ИНН за август–октябрь 2026, без отдельных операций. Файл `te_clients_2026_08_10.xlsx` рядом с тетрадкой, лист `клиенты`.

Слева колонки для поиска: ИНН, регион с названием, филиал, форма, размер бизнеса, код точки с названием и группой, ОКВЭД с названием, оборот, операции, чек, активные дни, место клиента по обороту среди всех и внутри своего кода точки, итог тестов (`разрезов_с_флагами`, `флагов`, `что_выбивается`; пусто, если клиент нигде не выбился или не проверялся), договоры, терминалы, точки, города, первая и последняя операция. Дальше оборот и операции по каждому месяцу, затем остальные поля профиля и карточки клиента.

В Excel включён фильтр и закреплены шапка и колонка ИНН. Чтобы найти клиентов региона, отфильтруйте `region` или `регион_название`: например, `35` Крым или `67` Севастополь. В тетрадке та же таблица — `clients_wide`.


In [ ]:
with imp:
    clients_wide = imp.fetch("""
select
  p.*,
  c.filial,
  c.ssp_ocrm,
  c.ogrn,
  c.jur_address,
  c.date_agr_current,
  t.agr_cnt,
  t.terminal_cnt,
  t.outlet_cnt,
  t.cities,
  t.first_trx,
  t.last_trx
from sandbox_ai.tmp_shestopalov_te_profile p
left join sandbox_ai.tmp_shestopalov_acq_clients c
  on c.inn = p.inn
left join (
  select
    inn,
    count(distinct n_agr) as agr_cnt,
    count(distinct id_pos_terminal) as terminal_cnt,
    count(distinct retl_id) as outlet_cnt,
    group_concat(distinct mrc_city, ' | ') as cities,
    min(date_t) as first_trx,
    max(date_t) as last_trx
  from sandbox_ai.tmp_shestopalov_acq_trx
  group by inn
) t
  on t.inn = p.inn
""")
    monthly = imp.fetch("""
select
  inn,
  substr(cast(date_t as string), 1, 7) as month,
  count(*) as tx_cnt,
  sum(sum_t) as turnover
from sandbox_ai.tmp_shestopalov_acq_trx
group by inn, substr(cast(date_t as string), 1, 7)
""")

monthly["turnover"] = pd.to_numeric(monthly["turnover"], errors="coerce")
month_turn = monthly.pivot_table(index="inn", columns="month", values="turnover", aggfunc="sum")
month_turn.columns = [f"оборот_{m}" for m in month_turn.columns]
month_tx = monthly.pivot_table(index="inn", columns="month", values="tx_cnt", aggfunc="sum")
month_tx.columns = [f"операций_{m}" for m in month_tx.columns]
clients_wide = clients_wide.merge(month_turn.reset_index(), on="inn", how="left")
clients_wide = clients_wide.merge(month_tx.reset_index(), on="inn", how="left")

clients_wide["turnover"] = pd.to_numeric(clients_wide["turnover"], errors="coerce")
clients_wide["mcc_code"] = clients_wide["mcc_code"].astype(str)
clients_wide["region"] = clients_wide["region"].astype(str)
clients_wide.insert(1, "регион_название", clients_wide["region"].map(lambda c: REGION_NAMES.get(c, "")))
clients_wide.insert(1, "mcc_название", clients_wide["mcc_code"].map(lambda c: MCC_NAMES.get(c.lstrip("0") or "0", "")))
clients_wide.insert(1, "оквэд_название", clients_wide["okved"].map(okved_label))
clients_wide["место_по_обороту"] = clients_wide["turnover"].rank(ascending=False, method="min").astype("Int64")
clients_wide["место_в_mcc"] = clients_wide.groupby("mcc_code")["turnover"].rank(ascending=False, method="min").astype("Int64")
clients_wide["клиентов_в_mcc"] = clients_wide.groupby("mcc_code")["inn"].transform("count")
clients_wide["доля_оборота_в_mcc"] = clients_wide["turnover"] / clients_wide.groupby("mcc_code")["turnover"].transform("sum")
clients_wide["inn"] = clients_wide["inn"].astype(str)
clients_wide = clients_wide.merge(flagged[["inn", "разрезов_с_флагами", "флагов", "что_выбивается"]], on="inn", how="left")
clients_wide = clients_wide.sort_values("turnover", ascending=False).reset_index(drop=True)

FRONT = [
    "inn", "region", "регион_название", "filial", "okopf", "business_size",
    "mcc_code", "mcc_название", "mcc_group", "okved", "оквэд_название",
    "turnover", "tx_count", "avg_amount", "median_amount", "active_days",
    "место_по_обороту", "место_в_mcc", "клиентов_в_mcc", "доля_оборота_в_mcc",
    "разрезов_с_флагами", "флагов", "что_выбивается",
    "agr_cnt", "terminal_cnt", "outlet_cnt", "cities", "first_trx", "last_trx",
]
front = [c for c in FRONT if c in clients_wide.columns]
months = sorted(c for c in clients_wide.columns if c.startswith(("оборот_20", "операций_20")))
rest = [c for c in clients_wide.columns if c not in front and c not in months]
clients_wide = clients_wide[front + months + rest]

print(f"Клиентов: {len(clients_wide)}. Колонок: {clients_wide.shape[1]}.")
crimea = clients_wide.loc[clients_wide["region"].isin(["35", "67"])]
print(f"Зарегистрированы в Крыму и Севастополе: {len(crimea)}.")
display(crimea[["inn", "region", "регион_название", "mcc_code", "mcc_название", "okved", "оквэд_название", "turnover", "tx_count", "место_по_обороту", "место_в_mcc", "cities"]])

out_clients = Path.cwd() / "te_clients_2026_08_10.xlsx"
try:
    with pd.ExcelWriter(out_clients) as writer:
        clients_wide.to_excel(writer, sheet_name="клиенты", index=False)
        sheet = writer.sheets["клиенты"]
        if hasattr(sheet, "auto_filter"):
            sheet.auto_filter.ref = sheet.dimensions
            sheet.freeze_panes = "B2"
    print("Сохранено:", out_clients)
except ImportError:
    out_csv = out_clients.with_suffix(".csv")
    clients_wide.to_csv(out_csv, index=False, encoding="utf-8-sig", sep=";")
    print("Сохранено:", out_csv)
